# Apartado 2 — Análisis de Árboles durante el Entrenamiento

En este apartado entrenamos un árbol de decisión utilizando el conjunto de datos
`music_classification.csv`. A diferencia del apartado 1, aquí **sí utilizamos** los modelos
ya implementados por scikit-learn, tal como indica el enunciado.

El objetivo es **analizar cómo cambian la estructura del árbol y sus métricas de rendimiento**
cuando modificamos:

- `min_samples_leaf` (tamaño mínimo de hojas)
- `ccp_alpha` (parámetro de complejidad para la poda de coste mínimo)

Los aspectos que debemos observar según el enunciado son:

1. El nodo raíz del árbol  
2. El tamaño total del árbol  
3. El número de hojas  
4. El F1-Score del modelo

El análisis debe realizarse fijando un hiperparámetro y variando el otro, y después
intercambiando los papeles de ambos.


In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

df = pd.read_csv("music_classification.csv")

print("Vista previa del dataset:")
display(df.head())

X = df.drop(columns=["Class"]).values
y = df["Class"].values


Vista previa del dataset:


,danceability,energy,key,loudness,mode,speechiness,acousticness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,0.854,0.564,1.0,-4.964,1,0.0485,0.017100,0.0849,0.8990,134.071,234596.0,4,5
1,0.382,0.814,3.0,-7.230,1,0.0406,0.001100,0.1010,0.5690,116.454,251733.0,4,10
2,0.434,0.614,6.0,-8.334,1,0.0525,0.486000,0.3940,0.7870,147.681,109667.0,4,6
3,0.853,0.597,10.0,-6.528,0,0.0555,0.021200,0.1220,0.5690,107.033,173968.0,4,5
4,0.167,0.975,2.0,-4.279,1,0.2160,0.000169,0.1720,0.0918,199.060,229960.0,4,10


## Función auxiliar para obtener información estructural del árbol

Para estudiar cada configuración de hiperparámetros, implementamos una función que:
- entrena el árbol,
- obtiene el nodo raíz (primer atributo testado),
- calcula el número total de nodos,
- calcula el número de hojas,
- calcula el F1-Score.

Esto nos permitirá comparar fácilmente las tendencias del modelo.


In [3]:
def analyze_tree(min_leaf, alpha):
    tree = DecisionTreeClassifier(
        min_samples_leaf=min_leaf,
        ccp_alpha=alpha,
        random_state=0
    ).fit(X, y)

    root_feature = tree.tree_.feature[0]
    n_nodes = tree.tree_.node_count
    n_leaves = tree.get_n_leaves()

    preds = tree.predict(X)
    f1 = f1_score(y, preds, average="macro")

    return {
        "min_samples_leaf": min_leaf,
        "ccp_alpha": alpha,
        "root_feature": root_feature,
        "n_nodes": n_nodes,
        "n_leaves": n_leaves,
        "f1_macro": f1
    }


## Experimento 1 — Fijar `ccp_alpha = 0` y variar `min_samples_leaf`

Aquí desactivamos la poda (`ccp_alpha = 0`) y modificamos el parámetro
`min_samples_leaf`, que controla el tamaño mínimo permitido en las hojas.

Valores probados:
- 1 (árbol sin restricciones → tiende a sobreajustar)
- 5
- 10
- 20

Observamos cómo cambia la estructura del árbol y el F1-Score.


In [4]:
results_exp1 = []

for m in [1, 5, 10, 20]:
    results_exp1.append(analyze_tree(min_leaf=m, alpha=0))

df_exp1 = pd.DataFrame(results_exp1)
df_exp1


,min_samples_leaf,ccp_alpha,root_feature,n_nodes,n_leaves,f1_macro
0,1,0,10,13173,6587,0.933019
1,5,0,10,5055,2528,0.728965
2,10,0,10,2667,1334,0.641513
3,20,0,10,1361,681,0.577053


## Interpretación del Experimento 1 — Variación de `min_samples_leaf` con `ccp_alpha = 0`

En este experimento hemos fijado `ccp_alpha = 0`, lo que implica **no realizar ninguna poda por complejidad**, y hemos variado el parámetro `min_samples_leaf` con los valores {1, 5, 10, 20}.


A partir de estos datos pueden extraerse las siguientes conclusiones:

---

### 1. Tamaño del árbol y sobreajuste

Cuando `min_samples_leaf = 1`, el árbol se expande sin ninguna restricción, alcanzando:

- **13.173 nodos**
- **6.587 hojas**

Este tamaño enorme indica **sobreajuste extremo**: el árbol memoriza prácticamente cada detalle del conjunto de entrenamiento.  
Esto se refleja en un **F1 alto (0.933)**, pero **solo porque se evalúa sobre el mismo conjunto usado para entrenar**, tal como se pide en el apartado.

---

### 2. Efecto de aumentar `min_samples_leaf`

A medida que incrementamos `min_samples_leaf`, el árbol se hace obligatoriamente más simple:

- Con `min_samples_leaf = 5`:  
  - Los nodos bajan a 5.055  
  - Las hojas bajan a 2.528  
  - El F1 cae a 0.729

- Con `min_samples_leaf = 10`:  
  - El árbol se reduce casi a la mitad  
  - F1 baja a 0.642

- Con `min_samples_leaf = 20`:  
  - El árbol queda en 1.361 nodos  
  - F1 cae a 0.577

Esto confirma lo esperado según Tema 7:

**Un valor mayor de `min_samples_leaf` fuerza hojas más grandes, reduce la profundidad del árbol, disminuye el sobreajuste y produce árboles más simples, pero también reduce la capacidad predictiva del modelo.**

---

### 3. El nodo raíz no cambia

El valor de `root_feature` es **siempre el mismo (índice 10)**.  
Esto significa que el atributo seleccionado en la raíz del árbol **es estable**, independientemente del valor de `min_samples_leaf`.

Este comportamiento concuerda con la teoría:  
la primera división se basa en la ganancia de información global y rara vez cambia al modificar restricciones estructurales.

---

### 4. Conclusión del experimento

- `min_samples_leaf = 1` produce un árbol **muy grande y sobreajustado**, con F1 aparente muy alto.  
- Al aumentar `min_samples_leaf`, el árbol se vuelve **más pequeño, más simple y menos profundo**, pero el F1 disminuye.  
- El efecto observado coincide exactamente con lo explicado en el **Tema 7** sobre técnicas de pre-poda.

Este experimento evidencia cómo este hiperparámetro controla la complejidad del árbol y su tendencia a sobreajustar.



# Experimento 2 — Fijar `min_samples_leaf = 1` y variar `ccp_alpha`

Ahora dejamos que cada hoja pueda tener un único ejemplo (`min_samples_leaf = 1`)
y aplicamos poda mediante el parámetro de complejidad `ccp_alpha`.

Valores probados:
- 0.0 (sin poda)
- 0.001
- 0.01
- 0.05

Observaciones teóricas (Tema 7):
- A mayor `alpha`, se podan ramas completas del árbol.
- El árbol se vuelve más simple.
- Puede mejorar o empeorar el rendimiento según la estructura del dataset.


In [5]:
results_exp2 = []

for a in [0.0, 0.001, 0.01, 0.05]:
    results_exp2.append(analyze_tree(min_leaf=1, alpha=a))

df_exp2 = pd.DataFrame(results_exp2)
df_exp2


,min_samples_leaf,ccp_alpha,root_feature,n_nodes,n_leaves,f1_macro
0,1,0.000,10,13173,6587,0.933019
1,1,0.001,10,61,31,0.445221
2,1,0.010,10,9,5,0.209823
3,1,0.050,-2,1,1,0.039216


## Interpretación del Experimento 2 — Variación de `ccp_alpha` con `min_samples_leaf = 1`

En este experimento mantenemos `min_samples_leaf = 1`, lo que permite que el árbol crezca hasta su máxima profundidad, y variamos únicamente el parámetro de poda por complejidad `ccp_alpha`.



A partir de estos datos pueden extraerse las siguientes conclusiones:

---

### 1. `ccp_alpha = 0`: árbol sin poda → sobreajuste extremo

Con `alpha = 0.000` obtenemos el mismo árbol del experimento 1 en el caso de `min_samples_leaf = 1`:

- **13.173 nodos**
- **6.587 hojas**
- **F1 = 0.933**

Este árbol memoriza prácticamente todo el conjunto de entrenamiento.  
La falta total de poda produce un modelo extremadamente grande y sobreajustado.

---

### 2. `ccp_alpha` pequeño → reducción drástica del tamaño del árbol

Con `alpha = 0.001` ocurre la primera poda significativa:

- El árbol pasa de **13.173 → 61 nodos**
- Las hojas de **6.587 → 31**
- El F1 cae a **0.445**

Esto confirma lo indicado en el Tema 7:

Valores pequeños de α ya son suficientes para eliminar ramas completas con baja ganancia de información.

Pese a que el árbol aún mantiene estructura, la reducción de tamaño es del **99.5%**.

---

### 3. `ccp_alpha` moderado → árboles muy pequeños

Con `alpha = 0.01`:

- El árbol cae a sólo **9 nodos**  
- Y **5 hojas**

El árbol ya es extremadamente simple y no puede capturar adecuadamente la complejidad del problema.

→ El F1 baja a **0.209**

---

### 4. `ccp_alpha` grande → poda excesiva = infraajuste total

Con `alpha = 0.05`:

- El árbol queda reducido a **1 nodo (raíz sin divisiones)**  
- Es una **hoja única** que predice siempre la clase mayoritaria  
- F1 se desploma a **0.039**

Esto representa el caso extremo de **infraajuste**.

---

### 5. El nodo raíz se mantiene estable hasta α muy grande

En los tres primeros casos:

- El atributo del nodo raíz sigue siendo **el mismo (índice 10)**.

Solo cuando `alpha = 0.05` el árbol queda reducido a una sola hoja y el root_feature pasa a -2, que en scikit-learn significa *“nodo hoja”*.

---

### 6. Conclusión general del experimento

Los resultados se alinean exactamente con la teoría del Tema 7:

- **ccp_alpha controla la complejidad del árbol mediante poda por coste mínimo**.
- Valores pequeños ya producen **poda efectiva**.
- Valores moderados producen **árboles simples y robustos**.
- Valores muy grandes producen **infraajuste severo**.

En resumen:

> **La poda por complejidad actúa como un regulador global del tamaño del árbol:**  
> - `alpha` bajo → más complejidad  
> - `alpha` alto → poda agresiva  
> - existe un punto intermedio donde se equilibra tamaño y rendimiento  

Este experimento demuestra de forma clara y numérica el efecto directo de `ccp_alpha` sobre la estructura del árbol y su capacidad predictiva.
